# UNITREE G1 · GR00T 全身控制（WBC + finetune 路线）

**目标**：用 NVIDIA 的 **GEAR-SONIC 全身控制器（WBC）当平衡底座**，再 finetune GR00T 让 G1 跳舞 —— 平衡交给鲁棒 WBC，VLA 只出高层 latent，**from construction 不摔**（对比我们 raw-joint ACT 蒸馏 0.5s 必摔）。

## 为什么走这条路
- 我们的 ACT 学生直接出 29-DoF 关节角 → covariate shift → 闭环 0.5s 摔。
- `UNITREE_G1_SONIC` 路线：VLA → SONIC **latent** → GEAR-SONIC WBC 解码 → 稳定全身关节。WBC 已会走/跑/跳，VLA 不用学精细平衡。

## 关键事实（已核实）
- `UNITREE_G1` / `UNITREE_G1_SONIC` 是 GR00T 的**动作空间配置（embodiment tag）**，不是现成模型；是 **posttrain tag**，要 finetune 才有 checkpoint。
- **没有现成微调好的 G1 VLA checkpoint**（GR00T-WholeBodyControl 的 VLA workflow 明确要从 `nvidia/GR00T-N1.7-3B` base 自己 finetune）→ 所以我们 **WBC + finetune**。
- 真正可下载的就两个：
  | 模型 | 是什么 | 能否预览 G1 |
  |---|---|---|
  | `nvidia/GEAR-SONIC` | WBC 底座（walk/run/jump），encoder/decoder/planner ONNX | ✅ 能（键盘交互 MuJoCo viewer） |
  | `nvidia/GR00T-N1.7-3B` | VLA「大脑」base | ❌ 单独不能（要 +WBC +finetune） |

调研详见 `doc/vla_data_expansion_covariate_shift.html` §6（架构侧 SONIC WBC 路线）。

## ① GEAR-SONIC WBC — 下载 + 安装（一次性，较重）

clone `NVlabs/GR00T-WholeBodyControl` → 装 MuJoCo sim 依赖（需 TensorRT + C++ build）→ 下载 GEAR-SONIC checkpoints。
首次较慢；若 install 报错按 repo README 处理 TensorRT。

In [ ]:
!bash scripts/gear_sonic_setup.sh

## ① 实时预览 GEAR-SONIC 行为（Isaac Lab eval，会走/踢/舞的 base）

单进程 Isaac Sim viewer 跑已发布的 SONIC WBC（`sonic_release/last.pt`），让它实时**跟踪**参考动作 —— **无 DDS、无 C++**。下面 cell 先看一条 walk，快速确认环境通。

**viewer 快捷键**（先点一下视口让它获得焦点）：
- `F` 自由相机（**停止自动跟随**，可自由拉远/旋转看全场）
- `R` 重置所有 env（动作重头播） · `T` 下一批 motion · `V` 参考骨架（绿点=目标姿态，看跟踪误差）。

In [ ]:
!bash scripts/gear_sonic_preview.sh

## ③ 一键看 SONIC 各动作（架构侧 Milestone 0 go/no-go）

本地 deploy demo 动作转成 `robot_filtered` 后，让 SONIC WBC 实时**跟踪**。**跟得住不摔 = 该动作活在 SONIC 的 token 空间**，后面 VLA 只要吐对应 64 维 token 就能驱动它（详见 `doc/groot_sonic_wbc_route.html`）。

每个 cell = 一个 G1 跑一条动作。重点看 **kick / dance / macarena** 摔不摔、跟不跟得上参考。
> 首次运行任意一条会自动建 `data/demo_robot_filtered.pkl`（7 条一起转，之后秒起）。关闭 viewer 窗口或 `Ctrl+C` 结束。

### viewer 快捷键（先点一下视口让它获得焦点）
| 键 | 作用 |
|---|---|
| **`F`** | **自由相机** —— 停止自动跟随机器人，可自由拉远/旋转/缩放看全场（解决"拉近就被复位"） |
| `R` | 重置所有 env，动作从头播 |
| `T` | 切换下一批 motion |
| `V` | 开关参考骨架可视化（绿点 = 目标姿态，肉眼看跟踪误差） |

> 默认是"相机跟随 env 0"模式，所以你拖拽视角下一帧会被拽回 —— **按 `F` 即可解除**。

In [ ]:
# 🕺 跳舞 dance_in_da_party
!bash scripts/gear_sonic_demo.sh dance

In [ ]:
# 🕺 macarena
!bash scripts/gear_sonic_demo.sh macarena

In [ ]:
# 🥋 踢腿 neutral_kick
!bash scripts/gear_sonic_demo.sh kick

In [ ]:
# 🦵 弓步 forward_lunge
!bash scripts/gear_sonic_demo.sh lunge

In [ ]:
# 🦘 单腿跳 one_leg_jumping
!bash scripts/gear_sonic_demo.sh jump

In [ ]:
# 🏋️ 深蹲 squat
!bash scripts/gear_sonic_demo.sh squat

In [ ]:
# 🚶 转身走 walking_quip_360
!bash scripts/gear_sonic_demo.sh walk

### 全部 7 条一排（每个 G1 各跳一条，paired 排开）

In [ ]:
# 7 个 G1 一排，各跳一条
!bash scripts/gear_sonic_demo.sh

## ② GR00T-N1.7-3B base VLA — 下载（finetune 用）

下载 VLA「大脑」base。**单独不能驱动 G1**（要配 SONIC WBC + finetune）。
gated：可能需 `huggingface-cli login` + 申请 access。

In [ ]:
!bash scripts/groot_n17_download.sh

## ④ 架构侧 VLA 自训 pipeline（record → finetune → 部署，全一键）

**思路**：GR00T N1.7 只输出 SONIC 的 64 维 FSQ token，WBC 当平衡底座解码成关节。
数据来自 SONIC 自己跟踪本地动作时录下的真 token（无需 VR），prompt 作为条件。

**链路**：`record(裸token npz)` → `convert(LeRobot v2.1)` → `finetune GR00T` → `open-loop eval(MSE)` → `dump 预测token` → `注入 SONIC 看动作`。

> 实测：8000 步后 open-loop token MSE **0.0011**（比 predict-mean 基线低 ~35×），踢腿/舞步注入后与 GT 基本同步 → 架构侧 GO（训练集内）。
> ⚠️ 现为 7-ep **过拟合**演示；**实时下任意 prompt + 泛化(held-out)= 下一步**。各 cell 较重（数据/GPU），按需跑。

### 4.1 录制 token 数据集（SONIC 跟踪 7 条动作，抽真 token+相机+proprio → 裸 npz）

In [ ]:
# headless 逐条录制，每条 1 episode（EPISODES=N 增量、SMOKE=1 快验）
!bash scripts/gear_sonic_record.sh

### 4.2 转 UNITREE_G1_SONIC LeRobot v2.1（官方 Gr00tDataExporter，需 .venv_data_collection）

In [ ]:
!cd dependencies/GR00T-WholeBodyControl && .venv_data_collection/bin/python \
  ../../scripts/sonic_vla_npz_to_lerobot.py \
  --raw-dir ../../datasets/sonic_vla_raw --out ../../datasets/sonic_vla_lerobot

### 4.3 finetune GR00T N1.7-3B（单 4090，冻结 VLM 训 DiT 头，adafactor+grad-ckpt，崩了自愈续训）

In [ ]:
# 全新 8000 步（让 LR schedule 铺满）；崩溃/重启自愈见 /tmp/resume_8k_heal.sh
!MAX_STEPS=8000 SAVE_STEPS=500 OUT_DIR=$PWD/outputs/gr00t_sonic_8k bash scripts/gr00t_sonic_finetune.sh

### 4.4 open-loop eval（预测 vs GT token 的 MSE，判 go/no-go；基线 per-motion-mean≈0.039）

In [ ]:
!cd dependencies/Isaac-GR00T && COMPILE_ACTION_HEAD_DISABLE=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  .venv/bin/python gr00t/eval/open_loop_eval.py \
  --model_path ../../outputs/gr00t_sonic_8k/checkpoint-8000 \
  --dataset_path ../../datasets/sonic_vla_lerobot \
  --embodiment_tag unitree_g1_sonic --action_horizon 40 --traj_ids 0 1 2 3 4 5 6 --steps 150

### 4.5 dump 预测 token（GR00T 对每条动作 open-loop 出 64 维 token → npz，供注入）

In [ ]:
!cd dependencies/Isaac-GR00T && COMPILE_ACTION_HEAD_DISABLE=1 \
  .venv/bin/python ../../scripts/gr00t_dump_pred_tokens.py \
  --model_path ../../outputs/gr00t_sonic_8k/checkpoint-8000 \
  --dataset_path ../../datasets/sonic_vla_lerobot --out ../../datasets/sonic_vla_pred_8k_final

### 4.6 注入 SONIC 看 GR00T 驱动的动作（默认用 checkpoint-8000 token + 放宽终止）
prompt→GR00T→token→SONIC 解码→G1 动，全内存无 C++。viewer：F 自由相机、V 参考、R 重置。

In [ ]:
# 🕺 dance — GR00T 预测 token 驱动
!bash scripts/gear_sonic_inject.sh dance

In [ ]:
# 🕺 macarena（最长 27.5s）
!bash scripts/gear_sonic_inject.sh macarena

In [ ]:
# 🥋 kick 踢腿
!bash scripts/gear_sonic_inject.sh kick

In [ ]:
# 🦵 lunge 弓步
!bash scripts/gear_sonic_inject.sh lunge

In [ ]:
# 🦘 jump 单腿跳
!bash scripts/gear_sonic_inject.sh jump

In [ ]:
# 🏋️ squat 深蹲
!bash scripts/gear_sonic_inject.sh squat

In [ ]:
# 🚶 walk 转身走
!bash scripts/gear_sonic_inject.sh walk

### 4.7 老师 vs 学生并排对比（左 env0=GT 真 token，右 env1=GR00T 预测 token，同跳一支）
两机器人基本同步 = GR00T 学到位（训练集内）。机器人本体无法逐 env 上色（Isaac instanceable 资产，去实例化会破物理），用左右位置区分。

In [ ]:
K=macarena_001__A545
GT=$PWD/datasets/sonic_vla_raw/$K/episode_000.npz
PR=$PWD/datasets/sonic_vla_pred_8k_final/$K.npz
!cd dependencies/GR00T-WholeBodyControl && DISPLAY=:0 conda run --no-capture-output -n isaaclab \
  python gear_sonic/eval_agent_trl.py +checkpoint=sonic_release/last.pt +headless=False ++num_envs=2 \
  ++manager_env.config.terrain_type=plane ++manager_env.config.env_spacing=3.0 \
  '++manager_env.config.viewer.eye=[0.0,-7.0,3.0]' '++manager_env.config.viewer.lookat=[0.0,0.0,1.0]' \
  ++manager_env.commands.motion.use_paired_motions=True \
  ++manager_env.commands.motion.motion_lib_cfg.motion_file=data/demo_robot_filtered.pkl \
  ++manager_env.commands.motion.motion_lib_cfg.filter_motion_keys=$K \
  ++manager_env.commands.motion.motion_lib_cfg.smpl_motion_file=dummy \
  ++manager_env.terminations.anchor_pos.params.threshold=99 ++manager_env.terminations.ee_body_pos.params.threshold=99 \
  ++manager_env.terminations.anchor_ori_full.params.threshold=99 ++manager_env.terminations.foot_pos_xyz.params.threshold=99 \
  '++eval_callbacks=[vla_injector]' \
  ++callbacks.vla_injector._target_=gear_sonic.data.vla_token_injector.VlaTokenInjector \
  "++callbacks.vla_injector.token_npz='$GT,$PR'"

## ⑤ 一键 live 演示流（时序 prompt 串接 · 自动起 server · 免 GUI/server 重启）

One running session, **the prompt switches over time** so the G1 does one motion after another
(squat → walk → macarena …) and loops — driven by a **live GR00T server** (闭环,非离线回放).

`gear_sonic_live_demo.sh` 自动**起 GR00T server**(若没起,加载约 1 分钟)再开 viewer,所以是真一键。
server 起一次可复用多个 flow;跑完用 **5.4 停止**释放显存。

> 数据结构 = `scripts/sonic_demo_flows.json`(演示流注册表)。`@flowN` 播注册的流,也可传 `squat,walk,kick` 这种即兴序列。

### 5.0 看有哪些演示流

In [ ]:
!bash scripts/gear_sonic_live_demo.sh list

### 5.1 flow1 — 离散动作循环（squat → lunge → macarena）
三个都是**自持动作**:纯 live GR00T 驱动,无 bootstrap,接缝最干净。

In [ ]:
!bash scripts/gear_sonic_live_demo.sh @flow1

### 5.2 flow2 — 动作 + walk 串接循环（squat → walk → macarena → walk → lunge → walk）
动作段 live GR00T;walk 段连续行走(full-bootstrap 回放走+转 clip)当连接器。

In [ ]:
!bash scripts/gear_sonic_live_demo.sh @flow2

### 5.3 即兴序列 + 视角/停止

**即兴序列**(逗号分隔,可 `名:步数`):

**viewer 快捷键**(先点一下视口获得焦点):`C` = 一键贴近机器人 · `F` = 跟随/自由切换 · `R` = 复位。

**视角微调**(relaunch 时加前缀):`CAM_EYE="x,y,z"` 相机位 · `CAM_LOOKAT="x,y,z"` 瞄准点
(lookat 第 3 位往下=多看腿脚;eye 前两位缩小=拉近)。每段时长 `STEPS=240`,接缝过渡 `SETTLE=40`。

In [ ]:
!bash scripts/gear_sonic_live_demo.sh squat,lunge,macarena

### 5.4 停止 live 演示（杀 viewer + GR00T server,释放显存）

In [ ]:
!bash scripts/gear_sonic_stop.sh

## 路线图：已完成 vs 下一步

**✅ 已跑通（见 ④）**：record → convert → finetune GR00T-N1.7（DiT 头出 64 维 SONIC token）→ open-loop MSE **0.0011**（破基线 35×）→ 注入 SONIC 解码 → 踢腿/舞步与 GT 基本同步。架构侧 GO（**训练集内**）。

**⏭️ 真正的下一步**：
1. **泛化（held-out）**：新 episode / 改写 prompt / 新起始姿态测 MSE —— 现在是 7-ep **过拟合**，没测过泛化。
2. **实时 prompt 活循环**：现演示是**开环回放**（离线 dump token 再注入）；要"敲 prompt → 机器人当场反应"，需 GR00T 在环 live `get_action`（in-process 或官方 `run_vla_inference.py` ZMQ）。
3. **数据多样性**：DR 放量近乎复制（实测 95% 重复）→ 真多样性靠**更多编舞 clip + RSI 相位**（Bones-SEED G1 CSV / LAFAN）。
4. **闭环纠偏**：开环长动作（如 macarena 27.5s）位置会漂；活循环里 GR00T 按实时 obs 出 token 可自纠。

> 数据侧便宜替代（DART/DAgger 治 raw-joint ACT covariate shift）见 `doc/vla_data_expansion_covariate_shift.html`。架构侧天花板高、架设重；数据侧便宜快。